# Tensor Indexing Memo

This memo formalizes tensor indexing using mathematical first principles. Brief outline
1. Establish the mental models of treating tensors as family of math functions.
2. Frame tensor indexing as function compositions.

## Tensor as family of math functions

A tensor is an n-dimensional array of primitive types. Common types are `torch.int16`, `torch.int32`, `torch.float16`, `torch.float32`, `torch.bool`. One can think of a tensor as a function mapping from n-dimensional integer indexes to a primitive value, e.g.:
* `t1: Float[torch.Tensor, 'batch seq d_model']` can be modeled as $t1: \mathbb{N_0}^3 \to \mathbb{R}$
* `t2: Bool[torch.Tensor, 'batch seq']` can be modeled as $t2: \mathbb{N_0}^2 \to \{0, 1\}$

But the codomain isn't limited to just primitives. In fact, codomains can be tensor functions too:
* `t1: Float[torch.Tensor, 'batch seq d_model']` can be modeled as $t1: \mathbb{N_0}^2 \to \mathbb{N_0} \times \mathbb{R}$, or as $t1: \mathbb{N_0} \to \mathbb{N_0}^2 \times \mathbb{R}$
* `t2: Bool[torch.Tensor, 'batch seq']` can be modeled as $t2: \mathbb{N_0} \to \mathbb{N_0} \times \{0, 1\}$

So a tensor type is really a family of math functions. And just like math functions, tensors can be "evaluated" by indexing at specific points to both (a) pick a math function from the family, and (b) evaluate that function at the given fixed point. For example, `t1[1, 0, 6]` means pick the math function $t1: \mathbb{N_0}^3 \to \mathbb{R}$, and plug in `(1, 0, 6)` to evaluate that function.

## Tensor as index - function composition

The function domain of a tensor ($\mathbb{N_0}^n$) is in itself a valid n-dimensional array of primitive type (in this case, `torch.int32`). Thus, The function domain can be though of as a tensor as well. This means we can perform function composition over a sequence of tensor functions, with the caveat that the tensor being used as an index *must*:
* have $\mathbb{N_0}^n$ as its domain, i.e. must be of type `Int[torch.Tensor, '...']`;
* the int values in the index tensor must be within range of the 0-th axis of the source tensor being indexed into;

Note the 2nd bullet point is subtle - why 0-th axis? Because when we write `source[index]`, by convention the `[.]` operator operates against the 0-th axis. This also implies the remaining axes of `source` are considerd as *output* of the indexing operation, aka *codomain* of the tensor after function composition. This echos the previous cell's notion that a tensor is a *family* of math functions, and in this case we happen to pick the family member $\mathbb{N_0} \to \mathbb{N_0}^{n-1} \times \mathbb{R}$.

Putting everything together: when we write `source[index]` in Python, mathematically this is really just a function composition:
* `source` tensor's function form is $\mathbb{N_0} \to \mathbb{N_0}^{n-1} \times \mathbb{R}$;
* `index` tensor's function form is $target: \mathbb{N_0}^m \to \mathbb{N_0}$;
* `source[index]` tensor's function form is $source \circ target$, which reduces to $\mathbb{N_0}^m \to \mathbb{N}^{n-1} \times \mathbb{R}$

This also imples `source[index].shape == (*index.shape, *source[1:].shape)`. There are no restrictions to shape of index or source. Only restrictions are the caveats noted above: index must be valued as int, and int values must be within range of source's 0-th axis.

In [19]:
import torch

# source.shape == (2, 3)
# source: N -> N x R
source = torch.Tensor([
    [1, 2, 3],
    [4, 5, 6],
]).to(torch.float32)

# index.shape == (2, 2, 3)
# index: N^3 -> N
index = torch.Tensor([
    # values must be within source's 0th axis values, i.e. < 2.
    [
        [1, 0, 1],
        [1, 1, 1],
    ],
    [
        [0, 0, 1],
        [0, 1, 1],
    ],
]).to(torch.int32)  # index tensor must be int.

# fuction composition: source[index].shape == (2, 2, 3, 3)
# source o index: N^3 -> R
assert source[index].shape == (2, 2, 3, 3)
expected = torch.Tensor([
    [
        [
            # index [1, 0, 1] -> pick source's 1st row, then 0th row, then 1st row
            [4, 5, 6],
            [1, 2, 3],
            [4, 5, 6],
        ],
        [
            # index [1, 1, 1] -> pick source's 1st row, then 1th row, then 1st row
            [4, 5, 6],
            [4, 5, 6],
            [4, 5, 6],
        ],
    ],
    [
        [
            # index [0, 0, 1] -> pick source's 0th row, then 0th row, then 1st row
            [1, 2, 3],
            [1, 2, 3],
            [4, 5, 6],
        ],
        [
            # index [0, 1, 1] -> pick source's 0th row, then 1th row, then 1st row
            [1, 2, 3],
            [4, 5, 6],
            [4, 5, 6],
        ],
    ],
]).to(torch.float32)
assert torch.equal(source[index], expected)